# Итоговый проект: шаблон пайплайна сбора данных

**Курс:** Автоматизированный сбор данных из интернета (код + no-code)

Это **рабочий каркас**, а не готовое решение. Инфраструктура (логирование, запись сырого слоя, отчёт о прогоне, самопроверка) уже реализована — её можно использовать как есть. Ваша работа — заполнить функции, помеченные `TODO: реализуйте`.

## Как работать с шаблоном

1. Заполните паспорт проекта и конфигурацию источников (разделы 0–1).
2. Реализуйте функции по порядку разделов. Каждая содержит контракт: что принимает, что возвращает, какие требования проверяются на защите.
3. Блокнот **исполняется целиком в любой момент**: нереализованные шаги не ломают прогон, а отмечаются как незавершённые (см. раздел 12 «Самопроверка»).
4. К защите все обязательные пункты самопроверки должны быть выполнены, а код — перенесён в модули `src/` (блокнот остаётся для демонстрации и аналитики).

**Запрещено:** обход капчи и средств защиты, маскировка, ротация IP/User-Agent для сокрытия, сбор из-под чужой авторизации, сбор с источников вне согласованного списка.


## 0. Паспорт проекта

Заполните перед началом работы: это первый слайд защиты.

In [ ]:
PROJECT_CARD = {
    "автор": "TODO: Фамилия И.О.",
    "вариант_конфигурации": "TODO: A (код) | B (гибрид) | C (no-code first)",
    "исследовательский_вопрос": "TODO: проверяемый вопрос, а не 'спарсить сайт X'",
    "почему_нужны_два_источника": "TODO: что даёт связь источников",
    "ожидаемый_объём_записей": None,      # TODO: int, >= 500
    "частота_обновления": "TODO: например, раз в сутки в 06:00 UTC",
    "контакт_в_user_agent": "TODO: mailto для администратора сайта",
    "есть_персональные_данные": None,     # TODO: True / False + обоснование в docs/legal_checklist.md
}

def check_card(card):
    problems = []
    for k, v in card.items():
        if v is None or (isinstance(v, str) and v.startswith("TODO")):
            problems.append(k)
    if problems:
        print("Не заполнено:", ", ".join(problems))
    else:
        print("Паспорт проекта заполнен.")
    return not problems

CARD_OK = check_card(PROJECT_CARD)

Не заполнено: автор, вариант_конфигурации, исследовательский_вопрос, почему_нужны_два_источника, ожидаемый_объём_записей, частота_обновления, контакт_в_user_agent, есть_персональные_данные


## 1. Окружение, конфигурация и служебная инфраструктура

Эти ячейки менять не обязательно: они создают структуру проекта, журнал и вспомогательные инструменты.

In [ ]:
import json, os, random, sqlite3, statistics, sys, time
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urljoin, urlparse

import pandas as pd
import requests

ROOT = Path("final_project")
DIRS = {k: ROOT / k for k in ["raw", "clean", "mart", "db", "logs", "reports", "state", "nocode", "docs"]}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
LOG_PATH = DIRS["logs"] / f"run_{RUN_ID}.jsonl"

CONTACT = PROJECT_CARD["контакт_в_user_agent"]
USER_AGENT = f"course-final-project/1.0 (+{CONTACT}; educational use)"

print("Каталоги проекта:", ", ".join(sorted(p.name for p in DIRS.values())))
print("Идентификатор прогона:", RUN_ID)
print("User-Agent:", USER_AGENT)
print("Python:", sys.version.split()[0], "| pandas:", pd.__version__)

Каталоги проекта: clean, db, docs, logs, mart, nocode, raw, reports, state
Идентификатор прогона: 20260922T103715Z
User-Agent: course-final-project/1.0 (+TODO: mailto для администратора сайта; educational use)
Python: 3.14.3 | pandas: 3.0.6


In [ ]:
def log_event(event, **fields):
    """Структурный журнал: одна строка — один JSON-объект. Текстовые логи на защите не принимаются."""
    record = {"ts": datetime.now(timezone.utc).isoformat(), "run_id": RUN_ID, "event": event}
    record.update(fields)
    with LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")
    return record


def not_implemented_names(*funcs):
    """Определяет, какие функции ещё являются заглушками."""
    import inspect
    pending = []
    for fn in funcs:
        try:
            src = inspect.getsource(fn)
        except OSError:
            continue
        if "raise NotImplementedError" in src:
            pending.append(fn.__name__)
    return pending


def safe_call(fn, *args, **kwargs):
    """Вызывает шаг пайплайна. Нереализованный шаг не прерывает прогон блокнота."""
    try:
        return fn(*args, **kwargs)
    except NotImplementedError as e:
        print(f"[НЕ РЕАЛИЗОВАНО] {fn.__name__}: {e}")
        log_event("step_not_implemented", step=fn.__name__)
        return None
    except Exception as e:
        print(f"[ОШИБКА] {fn.__name__}: {type(e).__name__}: {e}")
        log_event("step_failed", step=fn.__name__, error=f"{type(e).__name__}: {e}")
        return None


log_event("run_started", card_ok=CARD_OK)
print("Журнал прогона:", LOG_PATH)

Журнал прогона: final_project/logs/run_20260922T103715Z.jsonl


### 1.1. Конфигурация источников

Заполните «паспорт» каждого источника. Требование курса: минимум два источника — **один HTML-сайт и один REST API**. Адреса берите из файла «Источники данных для итогового проекта».

In [ ]:
SOURCES = {
    "html_catalog": {
        "тип": "html",
        "название": "TODO: например, books.toscrape.com",
        "точка_входа": "TODO: https://...",
        "пагинация": "TODO: page/N, кнопка 'next', бесконечная прокрутка, ...",
        "нужен_браузер": None,          # TODO: True/False + доказательство в разделе 2.2
        "ключ_уникальности": "TODO: какое поле источника образует бизнес-ключ",
        "лимит_запросов_в_сек": 1.0,
        "персональные_данные": None,    # TODO: True/False
        "условия_использования": "TODO: ссылка на ToS/robots.txt и вывод",
    },
    "api_source": {
        "тип": "api",
        "название": "TODO: например, Open Library",
        "точка_входа": "TODO: https://...",
        "пагинация": "TODO: limit/offset, cursor, Link-заголовок, page",
        "лимиты_из_документации": "TODO: заявленные ограничения",
        "ключ_уникальности": "TODO",
        "лимит_запросов_в_сек": 2.0,
        "персональные_данные": None,
        "условия_использования": "TODO",
    },
    # "dynamic_source": {...}   # добавьте, если в проекте есть браузерный сбор
}

pd.DataFrame(SOURCES).fillna("—")

,html_catalog,api_source
тип,html,api
название,"TODO: например, books.toscrape.com","TODO: например, Open Library"
точка_входа,TODO: https://...,TODO: https://...
пагинация,"TODO: page/N, кнопка 'next', бесконечная прокр...","TODO: limit/offset, cursor, Link-заголовок, page"
нужен_браузер,—,—
ключ_уникальности,TODO: какое поле источника образует бизнес-ключ,TODO
лимит_запросов_в_сек,1.0,2.0
персональные_данные,—,—
условия_использования,TODO: ссылка на ToS/robots.txt и вывод,TODO
лимиты_из_документации,—,TODO: заявленные ограничения


## 2. Правовая и техническая проверка источников

### 2.1. robots.txt и Crawl-delay

In [ ]:
def check_robots(entry_url: str, user_agent: str = USER_AGENT) -> dict:
    """TODO: реализуйте проверку правил сбора.

    Вход:  адрес точки входа и наш User-Agent.
    Выход: dict с ключами
           {"robots_url", "доступен" (bool), "can_fetch" (bool),
            "crawl_delay" (float|None), "request_rate" (str|None), "sitemaps" (list[str])}
    Подсказки: urllib.robotparser.RobotFileParser, методы read/can_fetch/crawl_delay/site_maps.
    Отсутствие robots.txt (404) не означает разрешения: проверьте условия использования.
    Проверяется на защите: заявленный Crawl-delay имеет приоритет над вашими настройками скорости.
    """
    raise NotImplementedError("раздел 2.1: проверка robots.txt")


for name, src in SOURCES.items():
    url = src["точка_входа"]
    if isinstance(url, str) and url.startswith("http"):
        print(name, "->", safe_call(check_robots, url))
    else:
        print(name, "-> точка входа не задана (TODO в разделе 1.1)")

html_catalog -> точка входа не задана (TODO в разделе 1.1)
api_source -> точка входа не задана (TODO в разделе 1.1)


### 2.2. Нужен ли браузер

Протокол диагностики из занятия 4 (выполняется **до** выбора Playwright):

1. Запросить страницу через `requests` и поискать в теле ответа фрагмент нужного текста.
2. Если текста нет — открыть DevTools → Network → XHR/Fetch и найти запрос, который приносит данные.
3. Если найден JSON-эндпоинт, собирать данные из него: это в разы быстрее и дешевле браузера.
4. Браузер оправдан, только если данных нет ни в HTML, ни в доступном API.

Результат диагностики (с цифрами) обязателен на защите.

In [ ]:
def diagnose_rendering(url: str, needle: str) -> dict:
    """TODO: реализуйте диагностику.

    Вход:  адрес страницы и фрагмент текста, который должен быть в данных.
    Выход: {"status", "bytes", "needle_in_html" (bool), "вывод": "достаточно HTTP" | "нужен браузер",
            "найденный_xhr": str|None}
    Требование: результат этой функции — аргумент в пользу выбранного инструмента, сохраните его в отчёт.
    """
    raise NotImplementedError("раздел 2.2: диагностика необходимости браузера")


safe_call(diagnose_rendering, SOURCES["html_catalog"]["точка_входа"], "TODO: искомый текст")

[НЕ РЕАЛИЗОВАНО] diagnose_rendering: раздел 2.2: диагностика необходимости браузера


## 3. Вежливый и устойчивый HTTP-клиент

Минимальные требования (занятие 7): лимит скорости по домену с джиттером, классификация ошибок, повторы только для временных сбоев, приоритет `Retry-After`, изоляция отказавшего домена.

In [ ]:
@dataclass
class RateLimiter:
    """TODO: реализуйте ограничитель по схеме token bucket.

    Параметры: rate (запросов/с), burst (размер всплеска), jitter (доля случайного разброса).
    Метод acquire() должен блокировать вызывающий код до момента, когда запрос разрешён.
    Требования: лимит задаётся ПО ДОМЕНУ; интервалы не должны быть строго одинаковыми (джиттер).
    """
    rate: float = 1.0
    burst: int = 1
    jitter: float = 0.25

    def acquire(self) -> None:
        raise NotImplementedError("раздел 3: token bucket")


LIMITERS: dict = {}   # TODO: заполните как {домен: RateLimiter(...)} по данным SOURCES


def classify(status: int | None, exc: Exception | None = None) -> tuple[str, bool]:
    """TODO: верните (класс_ошибки, повторять_ли).

    Обязательные случаи: 200, 304, 400, 401/403, 404, 429, 5xx, таймаут, ошибка соединения.
    Правило курса: 4xx (кроме 429) не повторяем; 429 — это просьба сбавить скорость, а не препятствие.
    """
    raise NotImplementedError("раздел 3: классификация ошибок")


def backoff_delay(attempt: int, base: float = 0.5, cap: float = 30.0) -> float:
    """TODO: экспоненциальная задержка с full jitter и ограничением cap."""
    raise NotImplementedError("раздел 3: backoff с джиттером")


def fetch(url: str, *, method: str = "GET", max_attempts: int = 4, **kwargs):
    """TODO: единая точка выхода в сеть.

    Обязанности:
      1) взять разрешение у RateLimiter нужного домена;
      2) отправить запрос с USER_AGENT и таймаутом;
      3) классифицировать результат, при временной ошибке — повторить (Retry-After важнее backoff);
      4) записать в журнал строку на каждую попытку: log_event("request", host=..., url=...,
         status=..., ms=..., attempt=..., bytes=..., error=...);
      5) вернуть Response или None, если исчерпаны попытки.
    Весь сетевой код проекта должен идти через эту функцию — иначе журнал будет неполным.
    """
    raise NotImplementedError("раздел 3: устойчивый fetch")


print("Заглушки раздела 3:", not_implemented_names(RateLimiter.acquire, classify, backoff_delay, fetch))

Заглушки раздела 3: ['acquire', 'classify', 'backoff_delay', 'fetch']


In [ ]:
class CircuitBreaker:
    """TODO: предохранитель на домен: closed -> open -> half-open -> closed.

    Параметры: порог последовательных отказов, время паузы.
    Поведение: в состоянии open запросы к домену не отправляются, остальные источники работают.
    Число активаций попадает в отчёт о прогоне.
    """
    def allow(self, host: str) -> bool:
        raise NotImplementedError("раздел 3: circuit breaker")

    def record(self, host: str, ok: bool) -> None:
        raise NotImplementedError("раздел 3: circuit breaker")


def conditional_headers(url: str) -> dict:
    """TODO: верните {'If-None-Match': ...} / {'If-Modified-Since': ...} из state/etags.json.

    Цель — получать 304 на неизменившихся ресурсах. Не забудьте сохранять ETag после успешного ответа.
    Требование проекта: не менее двух механизмов снижения нагрузки (условные запросы, инкрементальность,
    sitemap с lastmod, кэш при отладке, gzip) с измеренным эффектом.
    """
    raise NotImplementedError("раздел 3: условные запросы")


STATE_PATH = DIRS["state"] / "watermark.json"

def load_watermark() -> dict:
    return json.loads(STATE_PATH.read_text(encoding="utf-8")) if STATE_PATH.exists() else {}

def save_watermark(state: dict) -> None:
    STATE_PATH.write_text(json.dumps(state, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

print("Водяной знак:", load_watermark() or "пуст (первый прогон)")

Водяной знак: пуст (первый прогон)


## 4. Сбор: сырой слой

Правило занятия 6: сырой слой пишется **только дозаписью**, вместе с метаданными происхождения. Он позволяет перепарсить данные без повторного обращения к сайту — на защите об этом спросят.

In [ ]:
def write_raw(records, source_name: str) -> Path:
    """Готовая инфраструктура: запись сырых записей в JSON Lines с провенансом."""
    path = DIRS["raw"] / f"{source_name}.jsonl"
    n = 0
    with path.open("a", encoding="utf-8") as f:
        for rec in records:
            envelope = {
                "_run_id": RUN_ID,
                "_source": source_name,
                "_collected_at": datetime.now(timezone.utc).isoformat(),
                "_parser_version": "1.0.0",
                "payload": rec,
            }
            f.write(json.dumps(envelope, ensure_ascii=False, default=str) + "\n")
            n += 1
    log_event("raw_written", source=source_name, records=n, path=str(path))
    print(f"{source_name}: записано {n} сырых записей -> {path}")
    return path


def read_raw(source_name: str) -> list:
    """Готовая инфраструктура: чтение сырого слоя."""
    path = DIRS["raw"] / f"{source_name}.jsonl"
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


print("Сырой слой:", {p.name: len(read_raw(p.stem)) for p in DIRS["raw"].glob("*.jsonl")} or "пуст")

Сырой слой: пуст


In [ ]:
def collect_html(source: dict, limit: int | None = None) -> list[dict]:
    """TODO: сбор HTML-источника.

    Требования:
      - все запросы через fetch();
      - устойчивые селекторы (не зависящие от порядка и оформления);
      - корректная пагинация с остановкой (нет страницы -> нет запроса);
      - при необходимости переход на детальную страницу;
      - для books.toscrape.com: resp.encoding = "utf-8", иначе цены будут искажены;
      - возвращайте СЫРЫЕ поля, без очистки: чистка — в разделе 5.
    Выход: список dict с полями источника + "_url".
    """
    raise NotImplementedError("раздел 4: сбор HTML-источника")


def collect_api(source: dict, limit: int | None = None) -> list[dict]:
    """TODO: сбор API-источника.

    Требования: пагинация по документации источника; соблюдение заявленных лимитов;
    инкрементальность (водяной знак или ETag); нормализация вложенного JSON — в разделе 5.
    """
    raise NotImplementedError("раздел 4: сбор API-источника")


def collect_dynamic(source: dict, limit: int | None = None) -> list[dict]:
    """TODO (опционально): браузерный сбор, если диагностика раздела 2.2 доказала его необходимость.

    Внимание: в Jupyter работает только асинхронный Playwright:
        from playwright.async_api import async_playwright
        async with async_playwright() as p: ...
    Синхронный API в блокноте падает из-за уже запущенного цикла событий (в .py-скриптах он допустим).
    Ждите селектор (wait_for_selector), а не time.sleep; блокируйте изображения и шрифты.
    """
    raise NotImplementedError("раздел 4: браузерный сбор (если нужен)")

## 5. Очистка и контракт данных

In [ ]:
def clean_record(raw_envelope: dict, source_name: str) -> dict | None:
    """TODO: приведите одну сырую запись к единой схеме.

    Протокол очистки (занятие 6), примените осознанно и опишите в отчёте:
      1) нормализация пробелов и регистра, удаление невидимых символов;
      2) приведение типов (цена -> Decimal/float, дата -> ISO, количество -> int);
      3) единицы измерения и валюта к одному виду;
      4) справочные значения к словарю (категории, статусы);
      5) отсутствующие значения -> None (не пустая строка и не 0);
      6) вычисляемые поля и бизнес-ключ.
    Верните None, если запись бесполезна (нет ключевых полей) — она попадёт в rejects с причиной.
    """
    raise NotImplementedError("раздел 5: очистка записи")


def business_key(record: dict) -> str:
    """TODO: стабильный бизнес-ключ.

    Требование: НЕ URL и не порядковый номер страницы. Ключ должен выживать при смене адресов,
    сортировки и пагинации (артикул, ISBN, DOI, id источника, либо хэш от устойчивой пары полей).
    """
    raise NotImplementedError("раздел 5: бизнес-ключ")

In [ ]:
# TODO: опишите контракт данных. Пример каркаса на pydantic - замените поля своими.
try:
    from pydantic import BaseModel, Field, field_validator
    PYDANTIC = True
except ImportError:
    PYDANTIC = False
    print("pydantic не установлен: pip install pydantic")

if PYDANTIC:
    class Item(BaseModel):
        """TODO: минимум 6 полей, из них 3 обязательных, и не менее 2 собственных валидаторов."""
        key: str = Field(min_length=1)                 # бизнес-ключ
        title: str = Field(min_length=1, max_length=500)
        # TODO: price: float = Field(ge=0)
        # TODO: rating: int | None = Field(default=None, ge=1, le=5)
        # TODO: collected_at: datetime
        source: str

        # TODO: собственный валидатор №1 (например, согласованность наличия и количества)
        # @field_validator("title")
        # @classmethod
        # def _check_title(cls, v): ...

    print("Контракт данных объявлен. Не забудьте расширить поля и валидаторы.")


def validate_records(records: list[dict]) -> tuple[list[dict], list[dict]]:
    """TODO: разделите записи на валидные и отбракованные.

    Требование: каждая отбракованная запись сохраняется в clean/rejects.jsonl вместе с ПРИЧИНОЙ
    и исходной записью. Молча терять данные нельзя — на защите спросят долю и причины отбраковки.
    """
    raise NotImplementedError("раздел 5: валидация по контракту")

Контракт данных объявлен. Не забудьте расширить поля и валидаторы.


## 6. Хранилище и идемпотентная загрузка

In [ ]:
DB_PATH = DIRS["db"] / "warehouse.db"

SCHEMA_SQL = """
-- TODO: опишите схему под свои данные. Обязательно:
--   * PRIMARY KEY по бизнес-ключу (не по URL и не autoincrement);
--   * NOT NULL для обязательных полей, CHECK для диапазонов;
--   * служебные поля source, collected_at, updated_at, parser_version;
--   * индексы под запросы слоя mart.
CREATE TABLE IF NOT EXISTS items (
    key             TEXT PRIMARY KEY,
    title           TEXT NOT NULL,
    -- TODO: price REAL NOT NULL CHECK (price >= 0),
    -- TODO: rating INTEGER CHECK (rating BETWEEN 1 AND 5),
    source          TEXT NOT NULL,
    collected_at    TEXT NOT NULL,
    updated_at      TEXT NOT NULL,
    parser_version  TEXT NOT NULL
);
-- TODO: CREATE INDEX IF NOT EXISTS idx_items_source ON items(source);
"""

def init_db() -> sqlite3.Connection:
    conn = sqlite3.connect(DB_PATH)
    conn.executescript(SCHEMA_SQL)
    conn.commit()
    log_event("db_initialized", path=str(DB_PATH))
    return conn


def upsert_items(conn: sqlite3.Connection, records: list[dict]) -> dict:
    """TODO: идемпотентная загрузка через INSERT ... ON CONFLICT(key) DO UPDATE.

    Требования: повторный прогон не создаёт дублей; updated_at меняется ТОЛЬКО у реально изменившихся
    записей. Верните {"вставлено": n, "обновлено": n, "без изменений": n}.
    Демонстрация на защите: три прогона подряд с разными результатами (первый — вставка,
    второй — нули, третий — обновление изменённых записей).
    """
    raise NotImplementedError("раздел 6: UPSERT")


conn = init_db()
print("Схема инициализирована:", DB_PATH)
print("Таблицы:", [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'")])

Схема инициализирована: final_project/db/warehouse.db
Таблицы: ['items']


## 7. Метрики качества и пороги

In [ ]:
QUALITY_THRESHOLDS = {
    # TODO: обоснуйте каждый порог в отчёте; минимум шесть метрик
    "полнота обязательных полей, %": 99.0,
    "доля отбракованных записей, %": 5.0,
    "доля дублей по ключу, %": 0.0,
    "доля записей с пустой ценой, %": 2.0,
    "отклонение объёма от предыдущего прогона, %": 20.0,
    "доля ошибок HTTP, %": 5.0,
}

def quality_report(clean_records: list[dict], rejects: list[dict]) -> pd.DataFrame:
    """TODO: посчитайте метрики и сравните с порогами.

    Выход: DataFrame со столбцами ["метрика", "значение", "порог", "статус"],
    статус: ЗЕЛЁНЫЙ / ЖЁЛТЫЙ / КРАСНЫЙ. Правило проекта: при КРАСНОМ статусе загрузка в mart
    не выполняется, а прогон завершается с ненулевым кодом возврата.
    """
    raise NotImplementedError("раздел 7: метрики качества")


def detect_block(response) -> tuple[str, list[str]]:
    """TODO: детектор блокировки и «тихой» деградации.

    Учитывайте: код 403/429; текстовые маркеры (captcha, access denied, доступ ограничен, unusual traffic);
    аномально малый размер ответа; код 200 без ожидаемых элементов.
    Выход: (вердикт: "ok"|"подозрение"|"блокировка", список сработавших сигналов).
    При вердикте "блокировка": остановить сбор домена, сохранить тело ответа в logs/incidents/,
    записать инцидент в журнал. Обходить блокировку запрещено.
    """
    raise NotImplementedError("раздел 7: детектор блокировок")

## 8. Слой аналитики (mart) и выводы

In [ ]:
def build_mart(conn: sqlite3.Connection) -> dict:
    """TODO: постройте представления и агрегаты, отвечающие на ваш исследовательский вопрос.

    Требования: соединение данных двух источников; не менее двух агрегатов;
    выгрузка результата в mart/ (Parquet или CSV) для воспроизводимости.
    Выход: {"имя_представления": DataFrame}
    """
    raise NotImplementedError("раздел 8: слой mart")


def make_charts(marts: dict) -> None:
    """TODO: постройте не менее двух графиков по данным mart (matplotlib).

    Подписи осей и заголовки обязательны. Графики сохраните в mart/ или docs/.
    """
    raise NotImplementedError("раздел 8: графики")


FINDINGS = [
    "TODO: вывод 1 — утверждение + подтверждающие цифры",
    "TODO: вывод 2 — утверждение + подтверждающие цифры",
    "TODO: вывод 3 (при наличии)",
]
LIMITATIONS = [
    "TODO: чего данные не показывают (охват источника, период сбора, доля пропусков)",
    "TODO: возможные смещения выборки",
]
print("Выводы и ограничения должны быть заполнены до защиты.")

Выводы и ограничения должны быть заполнены до защиты.


## 9. Отчёт о прогоне и оповещения

Функция отчёта готова: она читает структурный журнал текущего прогона. Ваша задача — пороги и реакция на них.

In [ ]:
def run_report(log_path: Path = LOG_PATH) -> dict:
    """Готовая инфраструктура: агрегирует журнал прогона в метрики."""
    rows = [json.loads(l) for l in log_path.read_text(encoding="utf-8").splitlines() if l.strip()]
    req = [r for r in rows if r.get("event") == "request"]
    codes = {}
    for r in req:
        codes[str(r.get("status"))] = codes.get(str(r.get("status")), 0) + 1
    lat = sorted(r["ms"] for r in req if isinstance(r.get("ms"), (int, float)))
    def pct(p):
        return lat[min(len(lat) - 1, int(len(lat) * p))] if lat else None
    report = {
        "run_id": RUN_ID,
        "запросов": len(req),
        "коды": codes,
        "доля ошибок, %": round(100 * sum(1 for r in req if (r.get("status") or 500) >= 400) / len(req), 2) if req else 0.0,
        "доля 429/403, %": round(100 * sum(1 for r in req if r.get("status") in (403, 429)) / len(req), 2) if req else 0.0,
        "повторов": sum(1 for r in req if (r.get("attempt") or 1) > 1),
        "ответов 304": codes.get("304", 0),
        "латентность медиана, мс": statistics.median(lat) if lat else None,
        "латентность p95, мс": pct(0.95),
        "трафик, байт": sum(r.get("bytes") or 0 for r in req),
        "нереализованных шагов": sum(1 for r in rows if r.get("event") == "step_not_implemented"),
        "сбоев шагов": sum(1 for r in rows if r.get("event") == "step_failed"),
    }
    out = DIRS["reports"] / f"run_report_{RUN_ID}.json"
    out.write_text(json.dumps(report, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    return report


def alerts(report: dict) -> list[str]:
    """TODO: не менее четырёх порогов оповещения с обоснованием.

    Обязательно среди них: доля ошибок, доля 429/403, отклонение объёма данных от предыдущего прогона,
    отсутствие новых записей. Верните список текстов сработавших оповещений (пустой список — прогон здоров).
    В гибридной конфигурации оповещение отправляется через n8n.
    """
    raise NotImplementedError("раздел 9: пороги оповещения")


REPORT = run_report()
pd.Series(REPORT).to_frame("значение")

,значение
run_id,20260922T103715Z
запросов,0
коды,{}
"доля ошибок, %",0.0
"доля 429/403, %",0.0
повторов,0
ответов 304,0
"латентность медиана, мс",None
"латентность p95, мс",None
"трафик, байт",0


## 10. Единая точка входа

В сдаваемом проекте это `src/run_pipeline.py`, запускаемый из командной строки и по расписанию. Здесь — та же логика для демонстрации на защите.

In [ ]:
def run_pipeline(limit: int | None = 50) -> dict:
    """Каркас прогона: вызывает ваши функции и не падает на нереализованных шагах."""
    log_event("pipeline_started", limit=limit)
    stats = {}

    html_raw = safe_call(collect_html, SOURCES["html_catalog"], limit) or []
    api_raw = safe_call(collect_api, SOURCES["api_source"], limit) or []
    if html_raw:
        write_raw(html_raw, "html_catalog")
    if api_raw:
        write_raw(api_raw, "api_source")
    stats["собрано_html"], stats["собрано_api"] = len(html_raw), len(api_raw)

    cleaned = []
    for source_name in ("html_catalog", "api_source"):
        for env in read_raw(source_name):
            rec = safe_call(clean_record, env, source_name)
            if rec:
                cleaned.append(rec)
    stats["очищено"] = len(cleaned)

    valid_rejects = safe_call(validate_records, cleaned)
    valid, rejects = valid_rejects if valid_rejects else ([], [])
    stats["валидных"], stats["отбраковано"] = len(valid), len(rejects)

    quality = safe_call(quality_report, valid, rejects)
    blocked = quality is not None and "КРАСНЫЙ" in str(quality)
    if valid and not blocked:
        stats["загрузка"] = safe_call(upsert_items, conn, valid)
        stats["mart"] = list((safe_call(build_mart, conn) or {}).keys())
    else:
        print("Загрузка в mart пропущена: нет валидных данных или красный статус качества.")

    report = run_report()
    fired = safe_call(alerts, report) or []
    for a in fired:
        print("ОПОВЕЩЕНИЕ:", a)
    log_event("pipeline_finished", **{k: str(v) for k, v in stats.items()})
    return {"этапы": stats, "отчёт": report, "оповещения": fired}


RESULT = run_pipeline(limit=20)
print()
print("Итог прогона:", json.dumps(RESULT["этапы"], ensure_ascii=False, default=str))

[НЕ РЕАЛИЗОВАНО] collect_html: раздел 4: сбор HTML-источника
[НЕ РЕАЛИЗОВАНО] collect_api: раздел 4: сбор API-источника
[НЕ РЕАЛИЗОВАНО] validate_records: раздел 5: валидация по контракту
[НЕ РЕАЛИЗОВАНО] quality_report: раздел 7: метрики качества
Загрузка в mart пропущена: нет валидных данных или красный статус качества.
[НЕ РЕАЛИЗОВАНО] alerts: раздел 9: пороги оповещения

Итог прогона: {"собрано_html": 0, "собрано_api": 0, "очищено": 0, "валидных": 0, "отбраковано": 0}


## 11. No-code компонент

Обязателен хотя бы один работающий элемент: оркестрация (расписание, запуск, оповещение), сбор одного источника или выгрузка в таблицу. Экспорт workflow и скриншоты кладите в `nocode/`.

In [ ]:
NOCODE_MAPPING = pd.DataFrame([
    # TODO: заполните по своему проекту
    {"этап": "расписание",            "в коде": "TODO", "в no-code": "TODO: Schedule Trigger в n8n", "перенос": "TODO"},
    {"этап": "сбор источника",        "в коде": "TODO", "в no-code": "TODO", "перенос": "TODO"},
    {"этап": "повторы при ошибке",    "в коде": "TODO", "в no-code": "TODO: Retry On Fail", "перенос": "TODO"},
    {"этап": "контроль скорости",     "в коде": "TODO", "в no-code": "TODO: узел Wait / размер батча", "перенос": "TODO"},
    {"этап": "контракт данных",       "в коде": "TODO", "в no-code": "TODO", "перенос": "TODO: обычно не переносится"},
    {"этап": "метрики качества",      "в коде": "TODO", "в no-code": "TODO", "перенос": "TODO"},
    {"этап": "оповещение о сбое",     "в коде": "TODO", "в no-code": "TODO: Error Trigger + сообщение", "перенос": "TODO"},
])

NOCODE_COST = {
    "запусков в сутки": None,          # TODO
    "запросов/операций за запуск": None,  # TODO
    "операций в месяц": None,          # TODO
    "тариф и лимит платформы": "TODO",
    "вывод о целесообразности": "TODO",
}

NOCODE_MAPPING

,этап,в коде,в no-code,перенос
0,расписание,TODO,TODO: Schedule Trigger в n8n,TODO
1,сбор источника,TODO,TODO,TODO
2,повторы при ошибке,TODO,TODO: Retry On Fail,TODO
3,контроль скорости,TODO,TODO: узел Wait / размер батча,TODO
4,контракт данных,TODO,TODO,TODO: обычно не переносится
5,метрики качества,TODO,TODO,TODO
6,оповещение о сбое,TODO,TODO: Error Trigger + сообщение,TODO


## 12. Самопроверка готовности проекта

Запустите перед сдачей: таблица показывает, какие обязательные пункты ещё не выполнены.

In [ ]:
CHECKS = [
    ("Паспорт проекта заполнен", lambda: CARD_OK),
    ("Источники описаны (нет TODO в конфигурации)",
     lambda: not any(str(v).startswith("TODO") or v is None for s in SOURCES.values() for v in s.values())),
    ("robots.txt проверен", lambda: not not_implemented_names(check_robots)),
    ("Диагностика необходимости браузера", lambda: not not_implemented_names(diagnose_rendering)),
    ("Ограничитель скорости", lambda: not not_implemented_names(RateLimiter.acquire)),
    ("Классификация ошибок и backoff", lambda: not not_implemented_names(classify, backoff_delay)),
    ("Устойчивый fetch", lambda: not not_implemented_names(fetch)),
    ("Предохранитель", lambda: not not_implemented_names(CircuitBreaker.allow)),
    ("Условные запросы (ETag/Last-Modified)", lambda: not not_implemented_names(conditional_headers)),
    ("Сбор HTML-источника", lambda: not not_implemented_names(collect_html)),
    ("Сбор API-источника", lambda: not not_implemented_names(collect_api)),
    ("Сырой слой непуст", lambda: any(DIRS["raw"].glob("*.jsonl"))),
    ("Очистка записей", lambda: not not_implemented_names(clean_record)),
    ("Бизнес-ключ (не URL)", lambda: not not_implemented_names(business_key)),
    ("Контракт данных и валидация", lambda: not not_implemented_names(validate_records)),
    ("Схема БД заполнена", lambda: "TODO" not in SCHEMA_SQL),
    ("Идемпотентный UPSERT", lambda: not not_implemented_names(upsert_items)),
    ("Метрики качества с порогами", lambda: not not_implemented_names(quality_report)),
    ("Детектор блокировок", lambda: not not_implemented_names(detect_block)),
    ("Слой mart", lambda: not not_implemented_names(build_mart)),
    ("Графики (>= 2)", lambda: not not_implemented_names(make_charts)),
    ("Выводы по данным", lambda: not any(str(f).startswith("TODO") for f in FINDINGS[:2])),
    ("Ограничения работы указаны", lambda: not any(str(x).startswith("TODO") for x in LIMITATIONS)),
    ("Пороги оповещения", lambda: not not_implemented_names(alerts)),
    ("Структурный журнал пишется", lambda: LOG_PATH.exists() and LOG_PATH.stat().st_size > 0),
    ("Отчёт о прогоне сформирован", lambda: any(DIRS["reports"].glob("run_report_*.json"))),
    ("Сопоставление код / no-code", lambda: "TODO" not in NOCODE_MAPPING.to_string()),
    ("Оценка стоимости no-code", lambda: not any(v is None or str(v).startswith("TODO") for v in NOCODE_COST.values())),
]

rows = []
for name, test in CHECKS:
    try:
        ok = bool(test())
    except Exception as e:
        ok = False
    rows.append({"пункт": name, "статус": "готово" if ok else "не выполнено"})

readiness = pd.DataFrame(rows)
done = (readiness["статус"] == "готово").sum()
print(f"Выполнено {done} из {len(readiness)} пунктов ({100 * done // len(readiness)}%)")
print("Проект готов к сдаче только при 100%.")
readiness

Выполнено 2 из 28 пунктов (7%)


Проект готов к сдаче только при 100%.


,пункт,статус
0,Паспорт проекта заполнен,не выполнено
1,Источники описаны (нет TODO в конфигурации),не выполнено
2,robots.txt проверен,не выполнено
3,Диагностика необходимости браузера,не выполнено
4,Ограничитель скорости,не выполнено
5,Классификация ошибок и backoff,не выполнено
6,Устойчивый fetch,не выполнено
7,Предохранитель,не выполнено
8,Условные запросы (ETag/Last-Modified),не выполнено
9,Сбор HTML-источника,не выполнено


## 13. Правовой чек-лист

Скопируйте в `docs/legal_checklist.md` и заполните — он проверяется при приёме проекта.

| Пункт | Ваш ответ |
|---|---|
| Источники и их назначение | |
| Цель сбора и её правомерность | |
| Условия использования: есть ли запрет автоматического сбора | |
| `robots.txt`: релевантные правила, `Crawl-delay`, результат `can_fetch` | |
| Используется ли доступ из-под авторизации | |
| Применяются ли средства обхода защиты (должно быть «нет») | |
| Объём и частота запросов, пиковая нагрузка на домен | |
| Состав данных; есть ли персональные данные | |
| Правовое основание обработки (при наличии персональных данных) | |
| Место и срок хранения, порядок удаления | |
| Контакт для связи, указанный в `User-Agent` | |
| Действия при получении претензии или блокировки | |

## 14. Паспорт данных (data card)

Заполните `docs/data_card.md`: происхождение каждого поля, единицы измерения, частота обновления, покрытие и доля пропусков, известные ограничения и смещения, условия использования источника, контакт владельца проекта.

## 15. Подготовка к защите

**Регламент:** 7 минут доклад + 5 минут вопросы.

Подготовьте два сценария живой демонстрации:

1. **Инкрементальный прогон (30–60 с):** данные не изменились, видны ответы `304`, UPSERT возвращает нули, печатается отчёт о прогоне. Это доказывает идемпотентность и вежливость.
2. **Демонстрация устойчивости:** источник искусственно недоступен (`https://httpbin.org/status/503` или неверный адрес) — пайплайн изолирует отказ, пишет инцидент, завершается с ненулевым кодом и внятным сообщением.

Резервный вариант (записанные логи, отчёты, скриншоты) обязателен: сеть в аудитории может подвести.

**Готовьте ответы с цифрами на вопросы:** почему такой инструмент для источника; что является бизнес-ключом и почему не URL; что будет при обрыве прогона; сколько записей отбраковано и почему; как вы узнаете о смене вёрстки; какую нагрузку создаёте; как сократили число запросов; как реагируете на 429; есть ли персональные данные и на каком основании они обрабатываются; что перенесли бы в no-code, а что нет; какие ограничения у выводов; что изменится при росте объёма в 100 раз.

## 16. Перенос кода в проект

Блокнот — это стенд. В сдаваемой работе тот же код должен жить в модулях:

```
src/collect_html.py  collect_api.py  clean.py  contract.py
    storage.py  quality.py  observability.py  run_pipeline.py
```

`run_pipeline.py` запускается одной командой, читает `config.yaml`, пишет журнал и отчёт, возвращает ненулевой код при красном статусе качества. README должен позволить преподавателю воспроизвести прогон в чистом окружении.
